## FlatL2/FlatIP Index

In [2]:
import faiss
import numpy as np
d = 384   # embedding dimension
# IndexFlatIP -- exact search using Inner Product (cosine sim for normalised vectors)
index_flat = faiss.IndexFlatIP(d)
# Must normalise vectors BEFORE adding for IP to equal cosine similarity
vectors = np.random.rand(10000, d).astype("float32")
faiss.normalize_L2(vectors) # in-place normalisation
index_flat.add(vectors)
print(f"Vectors in index: {index_flat.ntotal}")  # 10000
# Query
query = np.random.rand(1, d).astype("float32")
faiss.normalize_L2(query)
k = 5
# return top-5 nearest neighbours
scores, indices = index_flat.search(query, k)
print("Top-5 indices:", indices[0])
print("Top-5 scores:", scores[0])   # cosine similarities (0 to 1 for normalised)

Vectors in index: 10000
Top-5 indices: [4403 3226 5412 8150 8309]
Top-5 scores: [0.81243074 0.80853415 0.80579996 0.8057356  0.80312955]


## IVFIndex - cluster based

In [4]:
import faiss
import numpy as np
d = 384
n = 100000  # 100k vectors
vectors = np.random.rand(n, d).astype("float32")
faiss.normalize_L2(vectors)
nlist = 200    # number of clusters -- rule of thumb: sqrt(n)
# IVFFlat -- IVF clustering + exact search within clusters
quantizer = faiss.IndexFlatIP(d)  
# used to assign vectors to clusters
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
# IVF requires training before adding vectors
assert not index_ivf.is_trained
index_ivf.train(vectors)   # learns the cluster centroids
assert index_ivf.is_trained
index_ivf.add(vectors)
print(f"Vectors in index: {index_ivf.ntotal}")
# nprobe controls the recall/speed tradeoff
index_ivf.nprobe = 10
# search 10 clusters out of 200
query = np.random.rand(1, d).astype("float32")
faiss.normalize_L2(query)
scores, indices = index_ivf.search(query, 5)
print("Results:", indices[0])
# Tuning nprobe: run searches at nprobe = 1, 5, 10, 50, 200
# and compare against IndexFlatIP results to measure recall at each setting
for nprobe in [1, 5, 10, 50, 200]:
    index_ivf.nprobe = nprobe
    _, ivf_results = index_ivf.search(query, 5)
    print(f"nprobe={nprobe:3d}: top result = {ivf_results[0][0]}")

Vectors in index: 100000
Results: [70877  6177 52026 30399 16884]
nprobe=  1: top result = 70877
nprobe=  5: top result = 70877
nprobe= 10: top result = 70877
nprobe= 50: top result = 77338
nprobe=200: top result = 77338


## HNSW Index

In [ ]:
import faiss
import numpy as np
d = 384
n = 100000
vectors = np.random.rand(n, d).astype("float32")
faiss.normalize_L2(vectors)
# HNSW index
M = 32
# connections per node -- higher = better recall, more memory
index_hnsw = faiss.IndexHNSWFlat(d, M, faiss.METRIC_INNER_PRODUCT)
# ef_construction: quality of the built graph
index_hnsw.hnsw.efConstruction = 200
# No training needed -- add directly
index_hnsw.add(vectors)
print(f"Vectors in index: {index_hnsw.ntotal}")
# ef_search: controls recall vs speed at query time
index_hnsw.hnsw.efSearch = 100
# increase for better recall, decrease for speed
query = np.random.rand(1, d).astype("float32")
faiss.normalize_L2(query)
scores, indices = index_hnsw.search(query, 5)
print("Results:", indices[0])
# HNSW does not support saving with faiss.write_index directly in all versions.
# Weaviate, Qdrant, and other databases manage HNSW persistence internally.
# Memory comparison (approximate):
# IndexFlatIP:n * d * 4 bytes = 100k * 384 * 4 = 153 MB
# IndexIVFFlat:similar to Flat + small centroid overhead
# IndexHNSWFlat:~2x Flat due to graph structure = ~300 MB for same data

## Metadata filtering with ChromaDB

In [ ]:
# pip install chromadb sentence-transformers
import chromadb
from huggingface_hub import InferenceClient
from sentence_transformers import SentenceTransformer
client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

model = "sentence-transformers/all-MiniLM-L6-v2"

client_cdb = chromadb.PersistentClient(path="./chroma_db") # saves to disk
# Create or load a collection
collection = client_cdb.get_or_create_collection(
name="university_docs",
metadata={"hnsw:space": "cosine"} # use cosine similarity
)
# Chunks with metadata
chunks = [
"Students must submit CNIC and transcripts for admission.",
"The semester fee is PKR 85,000 payable in two installments.",
"Late payment incurs a 5% penalty after the due date.",
"A full refund is available within the first two weeks.",
"The library is open Monday to Friday from 8am to 10pm.",
]
metadatas = [
{"source": "admission_policy.pdf", "section": "admission", "year": 2024},
{"source": "fee_structure.pdf","section": "fees","year": 2024},
{"source": "fee_structure.pdf","section": "fees","year": 2024},
{"source": "fee_structure.pdf","section": "refunds","year": 2024},
{"source": "hostel_policy.pdf","section": "library","year": 2023},
]
ids = [f"chunk_{i}" for i in range(len(chunks))]
embeddings = client.encode(chunks, normalize_embeddings=True).tolist()
# Add to collection
collection.add(
ids=ids,
embeddings=embeddings,
documents=chunks,
metadatas=metadatas
)
print(f"Total chunks in collection: {collection.count()}")
# ■■ Query WITHOUT filter ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
query = "what are the payment deadlines and penalties"
query_emb = client.encode([query], normalize_embeddings=True).tolist()
results = collection.query(query_embeddings=query_emb,
n_results=3
)
print("\nWithout filter:")
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
 print(f"[{meta['source']}] {doc[:60]}...")
# ■■ Query WITH metadata filter ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
# Only search within fee_structure.pdf
results_filtered = collection.query(
query_embeddings=query_emb,
n_results=3,
where={"source": "fee_structure.pdf"}
# metadata filter
)
print("\nFiltered to fee_structure.pdf only:")
for doc, meta in zip(results_filtered["documents"][0], results_filtered["metadatas"][0]):
 print(f"[{meta['section']}] {doc[:60]}...")
# ■■ Compound filter (AND) ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■
results_compound = collection.query(
query_embeddings=query_emb,
n_results=3,
where={"$and": [
{"source": {"$eq": "fee_structure.pdf"}},
{"year":
{"$gte": 2024}}
]}
)
print("\nFiltered to fee_structure.pdf AND year >= 2024:")
for doc in results_compound["documents"][0]:
 print(f"{doc[:60]}...")

## Metadata filtering with FAISS (post-filter approach)

In [ ]:
import faiss
import numpy as np
from huggingface_hub import InferenceClient
from sentence_transformers import SentenceTransformer
client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

model = "sentence-transformers/all-MiniLM-L6-v2"

chunks = [
"Students must submit CNIC and transcripts for admission.",
"The semester fee is PKR 85,000 payable in two installments.",
"Late payment incurs a 5% penalty after the due date.",
"A full refund is available within the first two weeks.",
"The library is open Monday to Friday from 8am to 10pm.",
]
metadata_store = {0: {"source": "admission_policy.pdf", "section": "admission"},
1: {"source": "fee_structure.pdf","section": "fees"},
2: {"source": "fee_structure.pdf","section": "fees"},
3: {"source": "fee_structure.pdf","section": "refunds"},
4: {"source": "hostel_policy.pdf","section": "library"},
}
text_store = {i: text for i, text in enumerate(chunks)}
embeddings = client.encode(chunks, normalize_embeddings=True).astype("float32")
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)
# POST-FILTER approach: over-retrieve, then filter by metadata
def filtered_search(query_text, filter_source=None, top_k=3, over_retrieve=20):
    query_emb = client.encode([query_text], normalize_embeddings=True).astype("float32")
    # Retrieve MORE than needed to compensate for post-filtering
    scores, indices = index.search(query_emb, over_retrieve)
    results = []
    for score, idx in zip(scores[0], indices[0]):
       if idx == -1: continue
       meta = metadata_store[idx]
         # Apply metadata filter
       if filter_source and meta["source"] != filter_source:
         continue
       results.append({"id": idx, "score": score,"text": text_store[idx], "meta": meta})
       if len(results) >= top_k:
         break
    return results
print("Filtered to fee_structure.pdf:")
for r in filtered_search("payment penalties", filter_source="fee_structure.pdf"):
   print(f"score={r['score']:.3f} | {r['text'][:60]}...")
# Save and reload index
faiss.write_index(index, "university.index")
loaded_index = faiss.read_index("university.index")
print("Index saved and reloaded successfully.")